# Weight demo

This notebook shows how to connect and drive a kern weight scale from Python.

## 1. Initialisation

In [139]:
import time
import csv
import threading
import numpy as np


from enderscope import WeightScale
from enderscope import Stage, ScanPatterns
from enderscope import Magnetometer, MagnetometerLogger
from pathlib import Path
from datetime import datetime

In [261]:
scale = WeightScale('COM11', baud_rate=9600)   # balance
ender = Stage('COM12', 115200)                   # plateau déplacement (X, Y, Z)
mag = Magnetometer('COM3', baud_rate=115200)    # magnétomètre (5 x MLX90393)

print(scale)
print(ender)
print(mag)

## 2. Positionnement / Force

### Homing et positionnement de la balance

<div style="background-color:#ffe0e0; border: 3px solid #d90000; border-radius: 8px; padding: 14px;">
<h2 style="color:#d90000; margin-top:0;">⚠️ ATTENTION ⚠️</h2>
<p style="font-size:16px; color:#7a0000;"><strong>RETIRER L'INDENTEUR avant de lancer le homing ci-dessous !</strong><br></p>
</div>

In [295]:
ender.home()
ender.set_speed(250)
ender.move_relative(0, 0, 60)
ender.finish_moves()

### Point de départ

Coordonnées du premier point de mesure à définit selon la manip (X, Y, Z)

In [299]:
ender.set_speed(150)
x_depart = 129 + 5
y_depart = 167 - 5
z_depart = 45

ender.move_absolute(x_depart, y_depart, z_depart)
ender.finish_moves()

### Tare de la balance

In [300]:
scale.perform_tare()

### Fonction d'indentation

In [ ]:
import msvcrt


def _stop_requested():
    if globals().get('STOP_DEMANDE', False):
        return True
    try:
        while msvcrt.kbhit():
            if msvcrt.getch() == b'\x1b':
                return True
    except OSError:
        pass
    return False


def apply_force(max_force=100, speed=150, mid_speed=250, max_retries=10, retry_delay=0.1,
                 overshoot_tol_g=10, max_corrections=5, max_depth_mm=30):
    force = 0
    ender.set_speed(speed)
    corrections = 0
    start_z = ender.get_position()[2]

    while True:
        if _stop_requested():
            raise KeyboardInterrupt

        reading = None
        for _ in range(max_retries):
            if _stop_requested():
                raise KeyboardInterrupt
            reading = scale.get_instant_reading()
            if reading is not None:
                break
            time.sleep(retry_delay)

        if reading is None:
            raise RuntimeError(
                f"Balance injoignable apres {max_retries} tentatives -- "
                f"arret par securite (derniere force connue : {force}g)"
            )
        force = reading

        current_z = ender.get_position()[2]
        profondeur = start_z - current_z
        if profondeur > max_depth_mm:
            raise RuntimeError(
                f"Profondeur ({profondeur:.2f} mm) au-dela du seuil de securite "
                f"({max_depth_mm} mm) -- arret immediat "
                f"(force actuelle : {force}g, cible : {max_force}g)"
            )

        if max_force <= force <= max_force + overshoot_tol_g:
            break

        if force > max_force + overshoot_tol_g:
            if corrections >= max_corrections:
                break
            corrections += 1
            ender.set_speed(int(speed / 2))
            ender.move_relative(0, 0, 0.1)
            if _stop_requested():
                raise KeyboardInterrupt
            ender.finish_moves()
            continue

        remaining = max_force - force
        if remaining > 200:
            ender.set_speed(speed)
            pas = -2
        elif remaining > 20:
            ender.set_speed(mid_speed)
            pas = -0.3
        else:
            ender.set_speed(int(speed / 2))
            pas = -0.1

        ender.move_relative(0, 0, pas)
        if _stop_requested():
            raise KeyboardInterrupt
        ender.finish_moves()

    if _stop_requested():
        raise KeyboardInterrupt
    time.sleep(0.15)

    reading = scale.get_instant_reading()
    if reading is not None:
        force = reading
    end_pos = ender.get_position()
    return force, end_pos


## 3. Magnétomètre

### Lecture brute

In [308]:
values = mag.get_reading()
print(values)

[6300.00048828125, 412.8000183105469, 127.7760009765625, 5059.2001953125, 1610.4000244140625, 3391.8720703125, -804.0000610351562, -5522.400390625, -2071.52001953125, -859.2000122070312, 4315.2001953125, -522.719970703125, 69.60000610351562, 2198.400146484375, 193.60000610351562]


### Baseline

Moyenne de quelques lectures au repos pour corriger les mesures.

In [311]:
baseline = mag.acquire_baseline(num_samples=5)
print('Baseline magnétomètre :', baseline)

Baseline magnétomètre : [ 6295.68017578   415.68001099   128.55039978  5057.76025391
  1614.24006348  3395.74389648  -801.60003662 -5520.48007812
 -2070.74560547  -858.24002686  4315.20019531  -522.7199707
    70.08000488  2188.32006836   195.92320557]


## 4. Acquisition & enregistrement CSV

### Dossier de sortie

Chaque prise de mesure crée son propre dossier, nommé par la date/heure et le type de mesure
(ex: `20260918_143210_mesure_combinee`). 

In [315]:
def create_output_dir(measurement_type: str) -> Path:
    timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = Path(f'{timestamp_str}_{measurement_type}') 
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir

### Plage de force

In [318]:
force_targets = np.arange(0, 2000, 50)
print(force_targets)

[   0   50  100  150  200  250  300  350  400  450  500  550  600  650
  700  750  800  850  900  950 1000 1050 1100 1150 1200 1250 1300 1350
 1400 1450 1500 1550 1600 1650 1700 1750 1800 1850 1900 1950]


### Motif de scan

Grille de positions autour du point de départ.

In [326]:
origin = [x_depart, y_depart] # coordinates of first position
steps = [5,-5] # x and y shifts to next item
positions = ScanPatterns.snake(6,6) * steps + origin
#ScanPatterns.plot_path(positions, field=(0,0),title='snake scan')

### a) Boucle de scan — mesure combinée (position + force)

1. création du dossier de sortie horodaté pour cette mesure
2. démarrage du `MagnetometerLogger` (flux indépendant, `sensor.csv` / `sensor_post_baselines.csv`)
3. boucle de scan : indentation à chaque position, écriture de `states.csv` avec son propre timestamp
4. fin du scan

In [ ]:
output_dir = create_output_dir('mesure_combinee')
print('Dossier de sortie :', output_dir)

mag_logger = MagnetometerLogger(
    mag, baseline,
    sensor_path=output_dir / 'sensor.csv',
    corrected_path=output_dir / 'sensor_post_baselines.csv',
)
mag_logger.start()

print("Scan en cours -- appuie sur Echap a tout moment pour arreter en securite.")

start_pos = None
try:
    with open(output_dir / 'states.csv', 'w', newline='') as f_states:
        writer_states = csv.writer(f_states)

        for scan_pos in positions:
            if _stop_requested():
                raise KeyboardInterrupt

            ender.move_position(scan_pos)
            ender.finish_moves()
            start_pos = ender.get_position()

            scale.perform_tare()

            for target_force in force_targets:
                if _stop_requested():
                    raise KeyboardInterrupt

                force, end_pos = apply_force(target_force)
                print(f'z: {end_pos[2]} - force: {force}g (cible: {target_force}g)')

                timestamp = time.time()
                x, y, z = end_pos
                writer_states.writerow([timestamp, x, y, z, force])

            ender.set_speed(150)
            ender.move_position(start_pos)
            ender.finish_moves()

except (KeyboardInterrupt, RuntimeError) as e:
    if isinstance(e, RuntimeError):
        print(f"\n\nArret de securite declenche -- {e}")
    else:
        print("\n\nEchap detecte -- arret d'urgence en cours...")
    ender.flush_serial_buffer()
    ender.write_code("M410", check_ok=False)  # Quickstop
    ender.flush_serial_buffer()
    print("Mouvement stoppe.")
    if start_pos is not None:
        print("Remontee en position de depart...")
        ender.set_speed(150)
        ender.move_position(start_pos)
        ender.finish_moves()
    print("Scan arrete.")

finally:
    mag_logger.stop()
    mag_logger.join()
    print("Scan termine.")


### b) Boucle de scan — mesure position seule
TODO

### c) Boucle de scan — mesure force seule
TODO

## Annexe : Test indentation — montee incrementale de la force jusqu'a une masse cible

Objectif : verifier si le silicone se dechire en approchant une force elevee (seuil souhaité 3000 g).


In [ ]:
FORCE_TARGET_MAX_G = 3000   # masse cible finale
FORCE_STEP_G = 200          # increment entre deux paliers
MAX_DEPTH_MM = 30           # profondeur relative max pour securite -- au-dela, arret auto

force_steps = list(range(FORCE_STEP_G, FORCE_TARGET_MAX_G + 1, FORCE_STEP_G))
print(f"Paliers prevus (g) : {force_steps}")

scale.perform_tare()
start_pos = ender.get_position()
z_depart = start_pos[2]

previous_depth = 0.0

try:
    for palier_g in force_steps:
        print(f"\n--- Palier cible : {palier_g} g ---")

        force, end_pos = apply_force(max_force=palier_g, speed=60)
        z = end_pos[2]
        profondeur = z_depart - z

        print(f"Force atteinte : {force:.1f} g (cible {palier_g} g) -- profondeur relative : {profondeur:.2f} mm")

        # Securite 1 : profondeur totale excessive
        if profondeur > MAX_DEPTH_MM:
            print(f"ATTENTION : profondeur ({profondeur:.2f} mm) au-dela du seuil de securite ({MAX_DEPTH_MM} mm).")
            print("Arret automatique -- le silicone a peut-etre cede (perte de resistance).")
            break

        # Securite 2 : saut anormal de profondeur par rapport au palier precedent
        delta = profondeur - previous_depth
        if previous_depth > 0 and delta > 3 * (MAX_DEPTH_MM / len(force_steps)):
            print(f"ATTENTION : saut de profondeur inhabituel entre ce palier et le precedent "
                  f"({delta:.2f} mm) -- possible perte de resistance du silicone.")
        previous_depth = profondeur

        reponse = input(
            f"Palier {palier_g} g atteint. Inspection du silicone, "
            f"puis appuyer sur Entree pour continuer, ou taper 'stop' pour arreter ici : "
        )
        if reponse.strip().lower() == 'stop':
            print("Arret demande par l'utilisateur.")
            break

except (KeyboardInterrupt, RuntimeError) as e:
    if isinstance(e, RuntimeError):
        print(f"\n\nArret de securite declenche -- {e}")
    else:
        print("\n\nEchap detecte -- arret d'urgence en cours...")
    ender.flush_serial_buffer()
    ender.write_code("M410", check_ok=False)  # Quickstop
    ender.flush_serial_buffer()
    print("Mouvement stoppe. Remontee en securite...")

print("\nRemontee en position de depart...")
ender.set_speed(6000)
ender.move_position(start_pos)
ender.finish_moves()
print("Test termine.")
